In [1]:
import itertools
import random
import unittest
import math
import scipy.stats as ss

import networkx as nx
import numpy as np
import skbio
import dendropy


In [2]:
def p_delta_change(n_states, l, change: bool):
    if not change:
        p_out = 1 / n_states + (n_states - 1) / n_states * math.exp(- (n_states - 1) * l)
    else:
        p_out = 1 / n_states - math.exp(- (n_states - 1) * l) / n_states
    return p_out


In [3]:
def simulate_cn_seq(prev_cn, n_states, l, alpha=1.):
    node_cn = np.empty_like(prev_cn)
    # scale l if needed
    pdd = p_delta_change(n_states, alpha * l, change=False)
    # simulate first copy number
    u = random.random()
    if u < pdd:
        node_cn[0] = prev_cn[0]
    else:
        node_cn[0] = random.choice([j for j in range(n_states) if j != prev_cn[0]])

    for m in range(1, len(prev_cn)):
        u = random.random()
        no_change_cn = prev_cn[m] - prev_cn[m-1] + node_cn[m-1]
        if 0 <= no_change_cn < n_states:
            if u < pdd:
                node_cn[m] = no_change_cn
            else:
                node_cn[m] = random.choice([j for j in range(n_states) if j != no_change_cn])
        else:
            node_cn[m] = random.choice([j for j in range(n_states)])
    return node_cn


In [4]:
def simulate_cn(tree, n_sites, n_states, alpha=1.):
    cn = np.empty((len(tree.nodes()), n_sites))
    cn[0, :] = 2
    # tree needs index-labeled node
    assert tree.seed_node.label is not None
    for n in tree.preorder_node_iter():
        if n.label != 0:
            cn[n.label] = simulate_cn_seq(cn[n.parent_node.label], n_states, n.edge_length, alpha=alpha)
    return cn


In [5]:
def emit_normalized_obs(cn_seq, mu=1.0, scale=1.0):
    eps = ss.norm(loc=0., scale=scale).rvs(size=len(cn_seq))
    return np.clip(cn_seq / mu + eps, a_min=0., a_max=None)


In [6]:
def label_tree(tree):
    for i, n in enumerate(tree.preorder_node_iter()):
        n.label = i


In [7]:
def rand_dataset(n_cells: int, n_states: int, n_sites: int, alpha=0.02, obs_type='norm') -> dict:
    # generate random sc binary tree
    tree = dendropy.treesim.treesim.birth_death_tree(.9, .4, num_extant_tips=n_cells)
    # ref: https://dendropy.org/primer/treesims.html
    # generate lengths
    # (done in dendropy)
    label_tree(tree)
    # simulate copy number chains
    # TODO: find best alpha for n_sites and delete argument from function (hide from out the function)
    cn = simulate_cn(tree, n_sites, n_states, alpha=alpha)
    # emit observations from tree leaves
    obs = np.empty((n_sites, n_cells))
    tax_id_map = {}
    for i, t in enumerate(tree.leaf_node_iter()):
        tax_id_map[t.taxon] = i
        if obs_type == 'pois':
            obs[:, i] = emit_raw_obs(cn[t.label])
        elif obs_type == 'norm':
            obs[:, i] = emit_normalized_obs(cn[t.label], scale=.7)
        else:
            logging.debug(f"type {obs_type} not supported for obs model")

    # return dict with observations and all latent variables
    data = {
        'obs': obs,
        'tree': tree,  # contains lengths as edges
        'cn': cn,
        'tax_id_map': tax_id_map
    }
    return data


In [8]:
data = rand_dataset(20, 8, 10)
tree = data['tree']


In [28]:
data.items()

dict_items([('obs', array([[2.84363695, 0.91021297, 2.72380942, 2.15760798, 1.4626117 ,
        2.07253424, 2.32876904, 3.60503143, 1.5585782 , 0.92926895,
        1.87388587, 2.34509595, 1.04550169, 1.53621478, 3.03261683,
        1.73891867, 0.71838399, 2.01711263, 1.51378021, 1.87832902],
       [1.9943199 , 1.7291014 , 1.29140409, 1.23645212, 2.9854272 ,
        2.17988385, 2.41373829, 2.78507229, 2.82265306, 2.91622804,
        1.04148271, 1.21058818, 6.47955557, 5.97983142, 0.73612814,
        0.98051701, 2.4080798 , 0.83940943, 0.93063993, 2.489171  ],
       [2.67152267, 1.70413954, 1.61746539, 5.76082902, 6.07629872,
        2.95962007, 3.04538357, 2.35549198, 1.94783608, 1.55075302,
        0.35773862, 0.        , 6.25923254, 5.26004354, 3.27737248,
        0.        , 1.3442177 , 3.89715252, 2.60062088, 2.3904697 ],
       [1.11069737, 2.08923131, 5.25126909, 5.92785115, 6.26859595,
        2.88360852, 1.76916832, 3.41173906, 2.53659403, 6.19956722,
        1.89176207, 0.776

In [14]:
pdc = tree.phylogenetic_distance_matrix()
for i, t1 in enumerate(tree.taxon_namespace[:-1]):
    for t2 in tree.taxon_namespace[i+1:]:
        print("Distance between '%s' and '%s': %s" % (t1.label, t2.label, pdc(t1, t2)))


Distance between 'T1' and 'T2': 3.1977344409217894
Distance between 'T1' and 'T3': 0.5359467390256096
Distance between 'T1' and 'T4': 3.1977344409217894
Distance between 'T1' and 'T5': 1.8295014577947084
Distance between 'T1' and 'T6': 1.1690877131844584
Distance between 'T1' and 'T7': 3.1977344409217894
Distance between 'T1' and 'T8': 3.1977344409217894
Distance between 'T1' and 'T9': 3.19773444092179
Distance between 'T1' and 'T10': 2.734176858499164
Distance between 'T1' and 'T11': 3.1977344409217894
Distance between 'T1' and 'T12': 3.1977344409217894
Distance between 'T1' and 'T13': 3.0804976829011403
Distance between 'T1' and 'T14': 1.8295014577947084
Distance between 'T1' and 'T15': 1.1690877131844584
Distance between 'T1' and 'T16': 3.1977344409217894
Distance between 'T1' and 'T17': 3.1977344409217894
Distance between 'T1' and 'T18': 1.8295014577947084
Distance between 'T1' and 'T19': 3.0804976829011403
Distance between 'T1' and 'T20': 3.0804976829011403
Distance between 'T2' a

In [15]:
pdc

In [16]:
type(tree)

dendropy.datamodel.treemodel._tree.Tree

In [17]:
nj_tree = pdc.nj_tree()

In [43]:
print(nj_tree.PhylogeneticDistanceMatrix())

AttributeError: 'Tree' object has no attribute 'PhylogeneticDistanceMatrix'

In [18]:
print(nj_tree.as_string("newick"))


[&U] (T9:0.7447226663442283,(((T19:0.19276079619919104,(T13:0.0127685335836597,T20:0.0127685335836597):0.17999226261553122):1.3474880452513773,(T10:1.3670884292495833,(((T15:0.025387733855148947,T6:0.025387733855148947):0.5591561227370799,(T1:0.2679733695128048,T3:0.2679733695128048):0.3165704870794247):0.330206872305125,(T5:0.6490379551651768,(T18:0.03479271677443794,T14:0.03479271677443794):0.6142452383907371):0.2657127737321785):0.45233770035222665):0.1731604122009902):0.1680402667827628,((T8:0.9028008066142625,(T12:0.7441921472225432,T17:0.7441921472225432):0.1586086593917202):0.4276060755418908,((T4:0.0,T16:0.0):0.9813206546014199,(T7:0.558522824941845,(T11:0.23487959493621882,T2:0.23487959493621882):0.3236432300056271):0.42279782965957535):0.34908622755473273):0.15903845053230353):0.7447226663442283);



In [27]:
pdc.patristic_distance(taxon1, taxon2, is_normalize_by_tree_size=False)

NameError: name 'taxon1' is not defined

In [31]:
vv = data['obs']

In [38]:
vv.shape

(10, 20)

In [34]:
print(vv)

[[2.84363695 0.91021297 2.72380942 2.15760798 1.4626117  2.07253424
  2.32876904 3.60503143 1.5585782  0.92926895 1.87388587 2.34509595
  1.04550169 1.53621478 3.03261683 1.73891867 0.71838399 2.01711263
  1.51378021 1.87832902]
 [1.9943199  1.7291014  1.29140409 1.23645212 2.9854272  2.17988385
  2.41373829 2.78507229 2.82265306 2.91622804 1.04148271 1.21058818
  6.47955557 5.97983142 0.73612814 0.98051701 2.4080798  0.83940943
  0.93063993 2.489171  ]
 [2.67152267 1.70413954 1.61746539 5.76082902 6.07629872 2.95962007
  3.04538357 2.35549198 1.94783608 1.55075302 0.35773862 0.
  6.25923254 5.26004354 3.27737248 0.         1.3442177  3.89715252
  2.60062088 2.3904697 ]
 [1.11069737 2.08923131 5.25126909 5.92785115 6.26859595 2.88360852
  1.76916832 3.41173906 2.53659403 6.19956722 1.89176207 0.77670514
  5.86342332 5.62082012 2.37163006 2.56295601 1.7933046  2.27713126
  1.97347977 1.98733926]
 [2.36242082 7.79207569 5.16500988 6.58519964 6.13313895 2.9571382
  4.34022105 1.79170403 1